# N5　RAG 維修知識庫：檢索與評估（第 8 節）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RealKazuya/ai-manufacturing-lab/blob/main/notebooks/N5_rag_retrieval.ipynb)

〈AI賦能製造：從工廠數據到AI決策〉課程筆記本。資料全部是教學用虛構（恆準精密工業）。

**用法**：由上往下逐格執行（Shift＋Enter）。要改的參數都在「參數」那一格。

In [ ]:
!pip -q install rank-bm25

In [1]:
# ── 環境設定（每次開啟先執行這一格）──
import sys, os
IN_COLAB = "google.colab" in sys.modules
LOCAL_DATA = os.path.join("..", "data")
DATA = LOCAL_DATA if os.path.isdir(LOCAL_DATA) else "https://raw.githubusercontent.com/RealKazuya/ai-manufacturing-lab/main/data/"
def path(p):
    return os.path.join(DATA, p) if DATA == LOCAL_DATA else DATA + p
print("資料來源：", "本機" if DATA == LOCAL_DATA else "GitHub", "｜Colab：", IN_COLAB)

資料來源： 本機 ｜Colab： False


## 參數

In [2]:
QUESTION = "VMC 主軸軸承鎖緊螺帽扭力是多少？"
CHUNK = "結構"        # 切塊方式："200"、"800"、"結構"
TOP_K = 5
USE_VECTOR = IN_COLAB  # Colab 上加向量檢索（multilingual-e5-small，CPU 約 1 分鐘）

## 讀手冊與工單

In [3]:
import pandas as pd, numpy as np, re, io, urllib.request
def read_text(p):
    return io.open(path(p), encoding="utf-8").read() if DATA == LOCAL_DATA else urllib.request.urlopen(path(p)).read().decode("utf-8")
manual = read_text("rag/manual.md")
manual = manual.split("-->", 1)[1] if manual.startswith("<!--") else manual
manual = re.sub(r"<!--.*?-->", "", manual)     # 拿掉排版用的註記
wo = pd.read_csv(path("rag/work_orders.csv"))
print("手冊 %d 字，工單 %d 筆" % (len(manual), len(wo)))

手冊 5336 字，工單 302 筆


## 切塊：三種切法

- 固定 200 字、固定 800 字：照字數硬切，表格可能被切成兩半。
- 依結構：一個小節一塊，表格整張不切。

每塊都有編號，生成時要附出處。

In [4]:
def sections(text):
    out, cur, title = [], [], "前言"
    for line in text.splitlines():
        if line.startswith("#"):
            if cur: out.append((title, "\n".join(cur).strip()))
            title, cur = line.lstrip("# ").strip(), []
        else:
            cur.append(line)
    if cur: out.append((title, "\n".join(cur).strip()))
    return [(t, b) for t, b in out if b]
def chunk_manual(how):
    if how == "結構":
        return [("手冊 " + t.split()[0], t + "\n" + b) for t, b in sections(manual)]
    n = int(how); flat = re.sub(r"\n+", "\n", manual)
    return [("手冊第 %d 段" % (i // n + 1), flat[i:i + n]) for i in range(0, len(flat), n)]
def chunk_wo():
    return [(r.wo_id, "%s %s %s 警報 %s；現象：%s；原因：%s；處置：%s；料號：%s；停機 %s 小時" % (r.wo_id, r.date, r.machine, r.alarm if isinstance(r.alarm, str) else "無", r.symptom, r.cause, r.action, r.part_no if isinstance(r.part_no, str) else "無", r.downtime_h)) for r in wo.itertuples()]
chunks = chunk_manual(CHUNK) + chunk_wo()
print("切成 %d 塊（手冊 %d、工單 %d）" % (len(chunks), len(chunks) - len(wo), len(wo)))

切成 325 塊（手冊 23、工單 302）


## 檢索：關鍵字（BM25）＋向量，混合排名

In [5]:
from rank_bm25 import BM25Okapi
def tok(s):   # 中文用相鄰兩字切詞，英數字整個保留
    s = s.lower(); words = re.findall(r"[a-z0-9\-\.]+", s)
    han = re.sub(r"[^\u4e00-\u9fff]", " ", s).split()
    return words + [h[i:i + 2] for h in han for i in range(max(1, len(h) - 1))]
bm25 = BM25Okapi([tok(c) for _, c in chunks])
vec = None
if USE_VECTOR:
    from sentence_transformers import SentenceTransformer
    emb_model = SentenceTransformer("intfloat/multilingual-e5-small")
    vec = emb_model.encode(["passage: " + c for _, c in chunks], normalize_embeddings=True, batch_size=64)
def search(q, k=TOP_K):
    s1 = bm25.get_scores(tok(q)); r1 = np.argsort(-s1)
    if vec is None: return [chunks[i] for i in r1[:k]]
    s2 = vec @ emb_model.encode(["query: " + q], normalize_embeddings=True)[0]; r2 = np.argsort(-s2)
    rrf = {}
    for r in (r1, r2):
        for rank, i in enumerate(r[:50]): rrf[i] = rrf.get(i, 0) + 1 / (60 + rank)
    return [chunks[i] for i, _ in sorted(rrf.items(), key=lambda x: -x[1])[:k]]
hits = search(QUESTION)
for cid, text in hits:
    print("【%s】%s\n" % (cid, text[:160].replace("\n", " ")))

【手冊 7.4】7.4 主軸軸承更換 主軸軸承更換標準工時 16 小時（含拆裝、預壓調整與試車），需兩人作業。  1. 完成 1.1 節上鎖掛牌。 2. 拆下主軸總成，置於清潔的工作台。 3. 拆下鎖緊螺帽與舊軸承，檢查軸承座有無磨損。 4. 以加熱器將新軸承加熱至 80 ℃ 後安裝，不得敲擊。 5. 依表 7-2 鎖緊螺帽扭力鎖緊，

【手冊 5.1】5.1 構造 主軸前側使用一組兩顆斜角滾珠軸承（料號 HZ-SB-7014），後側一顆（HZ-SB-7012），以鎖緊螺帽施加預壓。主軸外套有冷卻油道。

【手冊 9】9 備品清單 表 9-1　建議備品  | 料號 | 品名 | 數量 | 備註 | |---|---|---|---| | HZ-SB-7014 | 主軸軸承（前側，一組兩顆） | 1 組 | 日本原廠，經代理商進口 | | HZ-SB-7012 | 主軸軸承（後側） | 1 組 | 日本原廠，經代理商進口 | | HZ

【W-2026-0276】W-2026-0276 2026-05-12 VMC-05 警報 AL-2050；現象：主軸異音、振動過大；原因：主軸軸承劣化（外環剝離）；處置：依手冊 7.4 節更換主軸軸承並調整預壓，空轉振動合格後交機；料號：HZ-SB-7014；停機 19.0 小時

【W-2026-0252】W-2026-0252 2026-01-20 VMC-06 警報 AL-2050；現象：主軸異音、振動過大；原因：主軸軸承劣化（外環剝離）；處置：依手冊 7.4 節更換主軸軸承並調整預壓，空轉振動合格後交機；料號：HZ-SB-7014；停機 20.0 小時



## 組提示詞：複製下面整段，貼到 Claude

生成這一步刻意手動貼，看得到 LLM 到底收到什麼。規格數值要照抄出處，不准改寫。

In [6]:
prompt = "你是恆準精密的維修工程師助理。只根據下面的資料回答；資料裡沒有就回答「查無資料」。\n"
prompt += "規格數值（扭力、溫度、週期、料號）一律照抄原文，不換算、不改寫。每一句建議後面用【編號】標出處。\n\n"
prompt += "\n\n".join("【%s】\n%s" % (cid, text) for cid, text in hits)
prompt += "\n\n問題：" + QUESTION
print(prompt)

你是恆準精密的維修工程師助理。只根據下面的資料回答；資料裡沒有就回答「查無資料」。
規格數值（扭力、溫度、週期、料號）一律照抄原文，不換算、不改寫。每一句建議後面用【編號】標出處。

【手冊 7.4】
7.4 主軸軸承更換
主軸軸承更換標準工時 16 小時（含拆裝、預壓調整與試車），需兩人作業。

1. 完成 1.1 節上鎖掛牌。
2. 拆下主軸總成，置於清潔的工作台。
3. 拆下鎖緊螺帽與舊軸承，檢查軸承座有無磨損。
4. 以加熱器將新軸承加熱至 80 ℃ 後安裝，不得敲擊。
5. 依表 7-2 鎖緊螺帽扭力鎖緊，量測預壓。
6. 裝回主軸，依 5.2 節暖機，空轉量測振動低於 1.8 mm/s 方可交機。

表 7-2　主軸組裝扭力

| 項目 | 扭力 |
|---|---|
| 主軸軸承鎖緊螺帽 | 48 N·m |
| 主軸前蓋螺絲（M8） | 25 N·m |
| 主軸馬達聯軸器螺絲（M10） | 49 N·m |

【手冊 5.1】
5.1 構造
主軸前側使用一組兩顆斜角滾珠軸承（料號 HZ-SB-7014），後側一顆（HZ-SB-7012），以鎖緊螺帽施加預壓。主軸外套有冷卻油道。

【手冊 9】
9 備品清單
表 9-1　建議備品

| 料號 | 品名 | 數量 | 備註 |
|---|---|---|---|
| HZ-SB-7014 | 主軸軸承（前側，一組兩顆） | 1 組 | 日本原廠，經代理商進口 |
| HZ-SB-7012 | 主軸軸承（後側） | 1 組 | 日本原廠，經代理商進口 |
| HZ-SN-0048 | 主軸鎖緊螺帽 | 1 | 與軸承同時更換 |
| HZ-SE-1102 | 主軸編碼器 | 1 | 國內現貨 |
| HZ-LG-0201 | 主軸潤滑脂 | 1 支 | 國內現貨 |
| HZ-CO-0310 | 主軸冷卻油 | 20 L | 國內現貨 |
| HZ-DB-0410 | 拉刀碟形彈簧組 | 1 組 | 國內現貨 |
| HZ-FL-0120 | 冷卻機濾網 | 1 | 國內現貨 |
| HZ-BT-0005 | 絕對位置電池 | 1 | 國內現貨 |
| HZ-WP-0330 | 導軌刮屑片 | 1 組 | 國內現貨 |

【W-2026-0276】
W-2026-0276 2026-05-12 VMC

## 評估：黃金問題集

標準答案由人寫。看前 5 塊裡有沒有標準答案的出處（命中率@5）。

In [7]:
gold = pd.read_csv(path("rag/golden_reference.csv"))
def hit(row, how):
    global chunks, bm25
    chunks = chunk_manual(how) + chunk_wo(); bm25 = BM25Okapi([tok(c) for _, c in chunks])
    if row.type == "查無資料": return None
    got = " ".join(c + t for c, t in search(row.question))
    key = re.findall(r"表 [\d\-]+|\d+\.\d+ 節|\d+ 章", row.source)
    return any(k in got for k in key) or row.answer.split("，")[0].split("（")[0] in got
saved_vec, vec = vec, None          # 三種切法都只用關鍵字檢索比較（向量要重算太久）
rows = []
for how in ("200", "800", "結構"):
    r = [hit(g, how) for g in gold.itertuples()]
    ok = [x for x in r if x is not None]
    rows.append((how, "%d／%d" % (sum(ok), len(ok))))
chunks = chunk_manual(CHUNK) + chunk_wo(); bm25 = BM25Okapi([tok(c) for _, c in chunks]); vec = saved_vec
pd.DataFrame(rows, columns=["切塊方式", "命中@5（關鍵字檢索）"])

,切塊方式,命中@5（關鍵字檢索）
0,200,6／10
1,800,6／10
2,結構,7／10
